# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [6]:
# Write your code below.

%load_ext dotenv
%dotenv


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [7]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [14]:
import os
from glob import glob

# Write your code below.

price_data_path = os.getenv("PRICE_DATA")
print(price_data_path)

parquet_files = glob(os.path.join(price_data_path, "*.parquet"))
print(parquet_files[:5])

../../05_src/data/prices/
[]


In [ ]:
print(os.path.exists(price_data_path))  
print(os.listdir(price_data_path)[:10]) 


True
['ACN', 'AFTY', 'AGCO', 'AGM-A', 'ALDX', 'ALL', 'AMAL', 'AMH', 'AQMS', 'AVEM']


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [18]:
# Write your code below.

ddf = dd.read_parquet(price_data_path, engine="pyarrow").reset_index(drop=True)

# Lag features with explicit meta
ddf["Close_lag_1"] = ddf.groupby("ticker")["Close"].shift(1, meta=("Close", "f8"))
ddf["Adj_Close_lag_1"] = ddf.groupby("ticker")["Adj Close"].shift(1, meta=("Adj Close", "f8"))

# Returns
ddf["returns"] = (ddf["Close"] / ddf["Close_lag_1"]) - 1

# High - Low range
ddf["hi_lo_range"] = ddf["High"] - ddf["Low"]

# Keep only what we need
dd_feat = ddf[[
    "ticker", "Date", "Close", "Adj Close",
    "Close_lag_1", "Adj_Close_lag_1",
    "returns", "hi_lo_range"
]]



+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [25]:
# Write your code below.

import pandas as pd

# Instead of computing dd_feat (which breaks), reload parquet directly into Pandas
pdf = dd.read_parquet(price_data_path, engine="pyarrow").compute()

# Now feature engineering in Pandas (safe, no duplicate index problem)
pdf = pdf.sort_values(["ticker", "Date"]).reset_index(drop=True)

pdf["Close_lag_1"] = pdf.groupby("ticker")["Close"].shift(1)
pdf["Adj_Close_lag_1"] = pdf.groupby("ticker")["Adj Close"].shift(1)
pdf["returns"] = (pdf["Close"] / pdf["Close_lag_1"]) - 1
pdf["hi_lo_range"] = pdf["High"] - pdf["Low"]

# Add 10-day moving average of returns
pdf["ma_10_returns"] = pdf.groupby("ticker")["returns"].transform(lambda x: x.rolling(10).mean())

print(pdf.head(15))



         Date   Open   High    Low  Close  Adj Close      Volume   source  \
0  2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
1  2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
2  2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
3  2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
4  2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   
5  2001-07-26  14.95  14.99  14.50  14.50  10.900705   6335300.0  ACN.csv   
6  2001-07-27  14.51  14.59  14.50  14.51  10.908223   3524000.0  ACN.csv   
7  2001-07-30  14.50  14.78  14.50  14.70  11.051059   3654300.0  ACN.csv   
8  2001-07-31  14.71  15.01  14.60  14.96  11.246520   1429000.0  ACN.csv   
9  2001-08-01  15.00  15.50  14.90  15.50  11.652478   2087900.0  ACN.csv   
10 2001-08-02  15.40  15.50  15.10  15.40  11.577302   1717400.0  ACN.csv   
11 2001-08-03  15.40  15.40  14.98  15.15  11.389356    983600.0  ACN.csv   

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, it wasn’t strictly necessary to convert to pandas. Dask can do rolling windows, but it requires careful partitioning and handling overlaps
Yes, in this case it’s better to use pandas. The dataset fits in memory.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

In [ ]:
print(os.path.exists(price_data_path))  
print(os.listdir(price_data_path)[:10]) 


True
['ACN', 'AFTY', 'AGCO', 'AGM-A', 'ALDX', 'ALL', 'AMAL', 'AMH', 'AQMS', 'AVEM']


## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.